# NexusAI: обучение, чат и дообучение

Ноутбук рассчитан на GPU Colab. Он подключает Google Drive, клонирует проект с GitHub, скачивает ограниченные двуязычные подмножества Hugging Face в JSONL и предоставляет интерфейс для обучения/чата/дообучения. Для 1B–6B используйте готовые веса и подходящую GPU; обучение с нуля таких размеров в обычном Colab непрактично.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, pathlib, json
REPO_URL = 'https://github.com/OWNER/NexusAIprog.git'  # замените на URL проекта
WORK = '/content/NexusAIprog'
if not os.path.exists(WORK): subprocess.run(['git','clone',REPO_URL,WORK], check=True)
%cd /content/NexusAIprog
!pip -q install -r requirements.txt ipywidgets pandas pyarrow

In [ ]:
# Скачать 3 набора. max-bytes задаётся на каждый набор; 500 MB x 3 <= 2 GB.
!python scripts/prepare_datasets.py --output /content/drive/MyDrive/nexus_datasets --max-bytes 500000000

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path
size = widgets.Dropdown(options=['100M','300M','1B','3B','6B'], value='300M', description='Модель:')
steps = widgets.IntText(value=1000, description='Шаги:')
data = widgets.Text(value='/content/drive/MyDrive/nexus_datasets', description='Данные:', layout=widgets.Layout(width='700px'))
checkpoint = widgets.Text(value='', description='Готовые веса:', layout=widgets.Layout(width='700px'))
run = widgets.Button(description='Обучить / дообучить', button_style='success')
log = widgets.Output()
def train_clicked(_):
  with log:
    clear_output()
    cfg = f'configs/{size.value}.yaml'
    if checkpoint.value:
      import yaml
      c = yaml.safe_load(open(cfg))
      c['training']['resume_from'] = checkpoint.value
      c['training']['max_steps'] = steps.value
      yaml.safe_dump(c, open('/tmp/nexus_finetune.yaml','w'), allow_unicode=True)
      cfg = '/tmp/nexus_finetune.yaml'
    subprocess.run(['python','train.py','--config',cfg,'--data',data.value,'--max-length','512'], check=True)
run.on_click(train_clicked)
display(size, steps, data, checkpoint, run, log)

In [ ]:
# Чат с последним чекпойнтом
prompt = widgets.Text(value='Привет! Объясни, что ты умеешь.', description='Запрос:', layout=widgets.Layout(width='700px'))
ask = widgets.Button(description='Ответить')
chat_log = widgets.Output()
def ask_clicked(_):
  with chat_log:
    clear_output()
    subprocess.run(['python','inference.py','--checkpoint','checkpoints/latest.pt','--prompt',prompt.value,'--max-new-tokens','128'])
ask.on_click(ask_clicked)
display(prompt, ask, chat_log)

## Переключение размера при дообучении

В поле «Готовые веса» укажите чекпойнт, созданный с тем же размером модели и совместимым словарём. Выберите соответствующий YAML (300M/1B/3B/6B), задайте число шагов и нажмите кнопку. Результат сохраняется в `checkpoints/latest.pt`; скопируйте эту папку на Google Drive после обучения. Чекпойнт другого размера нельзя загрузить в архитектуру текущего размера без конвертации весов.